Submission by: Manasa Vuppu

# **Problem Statement:**

Title: "Network Ops" RAG Pipeline

Goal: Build a RAG tool that allows a Network Engineer to query a database of IP prefixes and services using natural language.

Context:
You are building a chatbot for a Cloud Network Operation Center (NOC). The bot needs to answer specific questions about which services own which IP ranges. Since this data changes weekly, the LLM's internal memory is outdated. You must rely on the provided JSON source.

1. The Data Source
File: AWS IP Address Ranges (Live JSON)
URL: https://ip-ranges.amazonaws.com/ip-ranges.json
Structure: It contains a large list under the key "prefixes".

Size: ~1.5 MB
Your Tasks

**Part 1: Ingestion **
Download the JSON file programmatically.
The "Ghost" Record: Before indexing, inject this fake critical record into the data (mimicking a private internal service):
{ "ip_prefix": "99.99.99.99/32", "region": "us-west-2", "service": "PROJECT_OMEGA_UPLINK", "network_border_group": "internal-restricted" }
Indexing: Parse the data and store it in a Vector Database
Explain in comments/readme what strategy was used and why?

**Part 2: The Retrieval Check**
Create a function debug_retrieval(question) that:

Searches the vector database for the question.
Prints the raw text of the Top-1 chunk.
Prints the Similarity Score.

**Part 3: The QA Chain **
Create a function ask_network_bot(question) that:

Retrieves the context.
Feeds it to an LLM (OpenAI/Anthropic/Local).
Returns the answer.

**Part 4: Evaluation (LLM-as-a-Judge)**
Run the following comparison script:

Question: "What is the IP address associated with Project Omega?"
System A (Zero-Shot): Ask the LLM directly.
System B (RAG): Ask your pipeline.
Judge: Write a simple logic (or LLM call) that checks:
If Answer contains "99.99.99.99" → PASS
Else → FAIL

**Deliverables**

A GitHub Gist or Repo link or .zip link containing:
files
README.md (Brief explanation of your approach)

SOLUTION APPROACH: This notebook implements a hybrid Retrieval-Augmented Generation (RAG) pipeline for querying AWS IP range data using natural language.

The system first downloads and parses the AWS IP ranges JSON file and injects a synthetic ghost record representing an internal service (PROJECT_OMEGA_UPLINK). Each record is converted into a structured text chunk and embedded using a SentenceTransformer model, then indexed in a FAISS vector database.

To improve reliability beyond pure vector search, the retrieval layer uses a hybrid strategy:

Deterministic structured retrieval via inverted indexes for service, region, ip_prefix, and network_border_group.

Constraint parsing from the user query (CIDR, IP address, region, service tokens).

AND-based filtering across matching structured fields.

Semantic FAISS fallback only when no structured constraints are present.

For IP queries, the system supports CIDR containment and Longest Prefix Match (LPM) using Python’s ipaddress module, enabling network-style prefix resolution.

The retrieved record is passed to a local LLM with a prompt instructing it to answer only from the provided context. A post-processing step extracts and returns the IP address in a clean format.

Finally, the notebook evaluates the system by comparing a zero-shot LLM baseline against the RAG-enhanced system, demonstrating improved factual accuracy for network lookup tasks.

In [179]:
!pip install faiss-cpu sentence-transformers

# Part 1: Ingestion

In [180]:
import json
import re
from typing import List, Dict, Any, Tuple, Optional

import requests
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import faiss

AWS_IP_RANGES_URL = "https://ip-ranges.amazonaws.com/ip-ranges.json"

GHOST_RECORD = {
    "ip_prefix": "99.99.99.99/32",
    "region": "us-west-2",
    "service": "PROJECT_OMEGA_UPLINK",
    "network_border_group": "internal-restricted",
}

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

In [181]:
def download_ip_ranges(url: str = AWS_IP_RANGES_URL, timeout_s: int = 30) -> Dict[str, Any]:
    r = requests.get(url, timeout=timeout_s)
    r.raise_for_status()
    return r.json()

def inject_ghost_record(data: Dict[str, Any], ghost: Dict[str, Any] = GHOST_RECORD) -> Dict[str, Any]:
    data = dict(data)  # shallow copy
    prefixes = list(data.get("prefixes", []))
    prefixes.insert(0, ghost)  # ensure included
    data["prefixes"] = prefixes
    return data

data = download_ip_ranges()
data = inject_ghost_record(data)

prefixes = data.get("prefixes", [])
print("Total prefixes (including ghost):", len(prefixes))
print("First record:", prefixes[0])

Total prefixes (including ghost): 9924
First record: {'ip_prefix': '99.99.99.99/32', 'region': 'us-west-2', 'service': 'PROJECT_OMEGA_UPLINK', 'network_border_group': 'internal-restricted'}


In [182]:
# --- Inspect downloaded structure ---

print("Top-level keys in JSON:")
print(data.keys())

print("\nTotal prefixes AFTER ghost injection:")
print(len(prefixes))

print("\nSample first 3 records:")
for i in range(3):
    print(prefixes[i])

Top-level keys in JSON:
dict_keys(['syncToken', 'createDate', 'prefixes', 'ipv6_prefixes'])

Total prefixes AFTER ghost injection:
9924

Sample first 3 records:
{'ip_prefix': '99.99.99.99/32', 'region': 'us-west-2', 'service': 'PROJECT_OMEGA_UPLINK', 'network_border_group': 'internal-restricted'}
{'ip_prefix': '3.4.12.4/32', 'region': 'eu-west-1', 'service': 'AMAZON', 'network_border_group': 'eu-west-1'}
{'ip_prefix': '3.5.140.0/22', 'region': 'ap-northeast-2', 'service': 'AMAZON', 'network_border_group': 'ap-northeast-2'}


In [183]:
def record_to_chunk(rec: Dict[str, Any]) -> str:
    # Canonical + labeled fields improves embedding stability for structured data
    return (
        f"service: {rec.get('service','')} | "
        f"ip_prefix: {rec.get('ip_prefix','')} | "
        f"region: {rec.get('region','')} | "
        f"network_border_group: {rec.get('network_border_group','')}"
    )

chunks: List[str] = [record_to_chunk(r) for r in prefixes]

**Chunking strategy**

1 record → 1 chunk (no splitting).

Each chunk is a canonical, flat text line with stable keys:
service | ip_prefix | region | network_border_group

**Why this approach:**

- Keeps structured tokens (service, IP Address, region) intact for matching
- Makes retrieval consistent + debuggable
- Avoids “semantic drift” you get when you over-chunk small structured rows

In [184]:
# Inspect first 5 chunks
for i in range(5):
    print(f"Chunk {i}:")
    print(chunks[i])
    print("-" * 80)

Chunk 0:
service: PROJECT_OMEGA_UPLINK | ip_prefix: 99.99.99.99/32 | region: us-west-2 | network_border_group: internal-restricted
--------------------------------------------------------------------------------
Chunk 1:
service: AMAZON | ip_prefix: 3.4.12.4/32 | region: eu-west-1 | network_border_group: eu-west-1
--------------------------------------------------------------------------------
Chunk 2:
service: AMAZON | ip_prefix: 3.5.140.0/22 | region: ap-northeast-2 | network_border_group: ap-northeast-2
--------------------------------------------------------------------------------
Chunk 3:
service: AMAZON | ip_prefix: 15.190.244.0/22 | region: ap-east-2 | network_border_group: ap-east-2
--------------------------------------------------------------------------------
Chunk 4:
service: AMAZON | ip_prefix: 15.230.15.29/32 | region: eu-central-1 | network_border_group: eu-central-1
--------------------------------------------------------------------------------


In production systems, ingestion should validate required fields (e.g., service, ip_prefix, region) and skip or log malformed records.

Deterministic retrieval systems should fail fast on incomplete structured data rather than silently degrade retrieval quality.

# Part 2: The Retrieval Check


In [185]:
# =========================
# LAYER 1: Deterministic Index Construction
# =========================
from collections import defaultdict
import re
from typing import Optional

def canon_service_or_nbg(s: Optional[str]) -> str:
    """
    Canonicalize service / network_border_group tokens for stable exact matching:
    - trim
    - uppercase
    - collapse internal whitespace
    NOTE: we do NOT try to "fuzzy match" here; that belongs in retrieval.
    """
    if not s:
        return ""
    s = s.strip().upper()
    s = re.sub(r"\s+", " ", s)  # collapse multiple spaces
    return s

def canon_region(s: Optional[str]) -> str:
    """
    Regions are typically lowercase like 'us-east-1'. Keep lowercase.
    """
    if not s:
        return ""
    return s.strip().lower()

def canon_ip_prefix(s: Optional[str]) -> str:
    """
    Keep CIDR prefixes verbatim except trimming whitespace.
    Do NOT normalize/uppercase IPs.
    """
    if not s:
        return ""
    return s.strip()

service_to_idxs = defaultdict(list)
prefix_to_idxs  = defaultdict(list)
region_to_idxs  = defaultdict(list)
nbg_to_idxs     = defaultdict(list)

for i, rec in enumerate(prefixes):
    svc     = canon_service_or_nbg(rec.get("service"))
    ip_pref = canon_ip_prefix(rec.get("ip_prefix"))
    reg     = canon_region(rec.get("region"))
    nbg     = canon_service_or_nbg(rec.get("network_border_group"))

    if svc:
        service_to_idxs[svc].append(i)
    if ip_pref:
        prefix_to_idxs[ip_pref].append(i)
    if reg:
        region_to_idxs[reg].append(i)
    if nbg:
        nbg_to_idxs[nbg].append(i)

# Allowlists for later gating / "unknown constraint" behavior
KNOWN_SERVICES = set(service_to_idxs.keys())
KNOWN_REGIONS  = set(region_to_idxs.keys())
KNOWN_NBGS     = set(nbg_to_idxs.keys())
KNOWN_PREFIXES = set(prefix_to_idxs.keys())

print(
    "Exact index sizes:",
    "services =", len(service_to_idxs),
    "| ip_prefix =", len(prefix_to_idxs),
    "| regions =", len(region_to_idxs),
    "| network_border_group =", len(nbg_to_idxs),
)

Exact index sizes: services = 27 | ip_prefix = 7432 | regions = 42 | network_border_group = 114


In [186]:
print(sorted(service_to_idxs.keys()))

['AMAZON', 'AMAZON_APPFLOW', 'AMAZON_CONNECT', 'API_GATEWAY', 'AURORA_DSQL', 'CHIME_MEETINGS', 'CHIME_VOICECONNECTOR', 'CLOUD9', 'CLOUDFRONT', 'CLOUDFRONT_ORIGIN_FACING', 'CODEBUILD', 'DYNAMODB', 'EBS', 'EC2', 'EC2_INSTANCE_CONNECT', 'GLOBALACCELERATOR', 'IVS_LOW_LATENCY', 'IVS_REALTIME', 'KINESIS_VIDEO_STREAMS', 'MEDIA_PACKAGE_V2', 'PROJECT_OMEGA_UPLINK', 'ROUTE53', 'ROUTE53_HEALTHCHECKS', 'ROUTE53_HEALTHCHECKS_PUBLISHING', 'ROUTE53_RESOLVER', 'S3', 'WORKSPACES_GATEWAYS']


**Layer 1: Deterministic Structured Retrieval**

This layer builds inverted indexes over structured fields (service, ip_prefix, region, network_border_group) to enable fast, exact lookup before semantic retrieval.

Each field value maps to the set of records containing that value, allowing the retrieval layer to quickly obtain candidate records for structured constraints extracted from the user query.

This design is important because embeddings are not reliable for precise identifiers such as service names or CIDR prefixes. Exact indexing ensures correctness, efficient lookup, and enables multi-constraint filtering without semantic ambiguity.

In [187]:
# =========================
# LAYER 2: Semantic Retrieval (Embeddings + FAISS)
# =========================
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

if not chunks or len(chunks) == 0:
    raise ValueError("`chunks` is empty. Cannot build FAISS index.")

# Encode -> normalize (cosine-ready) -> float32 for FAISS
embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)
embeddings = np.asarray(embeddings, dtype=np.float32)

dim = int(embeddings.shape[1])
index = faiss.IndexFlatIP(dim)  # inner product == cosine when vectors are normalized
index.add(embeddings)

print("FAISS index size:", index.ntotal, "| dim:", dim)

def faiss_search(query: str, k: int = 5):
    """
    Returns (idxs, scores) for top-k results.
    Scores are cosine similarities because embeddings are normalized.
    """
    q = embedder.encode([query], normalize_embeddings=True)
    q = np.asarray(q, dtype=np.float32)
    scores, idxs = index.search(q, k)
    return idxs[0].tolist(), scores[0].tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/311 [00:00<?, ?it/s]

FAISS index size: 9924 | dim: 384


**Layer 2: Semantic Retrieval (Embeddings + FAISS)**

This layer builds a vector index over the canonical text chunks to support semantic retrieval. Each chunk is embedded using a SentenceTransformer model (all-MiniLM-L6-v2) with L2 normalization enabled, and the resulting vectors are stored in a FAISS IndexFlatIP.

Because vectors are normalized, inner product search is equivalent to cosine similarity, enabling efficient similarity search over the full dataset. The FAISS index is in-memory and provides fast top-k nearest neighbor retrieval.

In this notebook, semantic retrieval is used as a fallback mode for queries that do not contain strong structured identifiers (e.g., CIDR/IP, explicit service token, region, or network_border_group). Structured queries are handled deterministically to avoid semantic mismatches (e.g., returning an unrelated service due to embedding similarity).

**Production Upgrade:** From In-Memory FAISS to a Persistent Vector Store

FAISS is good for low-latency local prototyping, but it is not persistent and does not provide database features like durability, access control, or metadata filtering at scale.

Managed option (Pinecone / similar): supports horizontal scaling, namespaces (e.g., separate network zones), and metadata filtering (e.g., filter by region/service before vector search).

Self-hosted option (Postgres + pgvector): allows storing structured AWS IP fields and embeddings together, enabling hybrid SQL + vector search in a single system (useful when the source data is naturally relational).

In [188]:
# =========================
# LAYER 3: Hybrid Retrieval Logic (Exact → Vector Fallback)
# =========================
import re
import difflib
import ipaddress
from dataclasses import dataclass
from typing import Optional, Set, List, Tuple

IP_RE     = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
CIDR_RE   = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}/\d{1,2}\b")
REGION_RE = re.compile(r"\b(?:us|eu|ap|sa|ca|me|af)-(?:north|south|east|west|central|northeast|southeast|northwest|southwest)-\d\b", re.IGNORECASE)
NBG_RE    = re.compile(r"\binternal-restricted\b", re.IGNORECASE)

def norm_service_key(s: str) -> str:
    """
    Normalization that fixes spacing/underscore/hyphen/punctuation differences:
      "route 53" / "ROUTE-53" / "route_53" -> "ROUTE53"
    """
    s = (s or "").strip().upper()
    s = re.sub(r"[^A-Z0-9]+", "", s)   # drop all separators
    return s

def norm_region_key(s: str) -> str:
    return (s or "").strip().lower()

def norm_nbg_key(s: str) -> str:
    return (s or "").strip().lower()

KNOWN_SERVICES = globals().get("KNOWN_SERVICES", set(service_to_idxs.keys()))
KNOWN_REGIONS  = globals().get("KNOWN_REGIONS", set(region_to_idxs.keys()))
KNOWN_NBGS     = globals().get("KNOWN_NBGS", set(nbg_to_idxs.keys()))
KNOWN_PREFIXES = globals().get("KNOWN_PREFIXES", set(prefix_to_idxs.keys()))

NORM_TO_SERVICE = {norm_service_key(svc): svc for svc in KNOWN_SERVICES}

def resolve_service_token(raw: str, fuzzy_cutoff: float = 0.92) -> Tuple[Optional[str], List[str]]:
    """
    Resolve a user-provided service token to a known service.

    Returns:
      (resolved_service_or_None, suggestions)
    Safety:
      - exact normalized match first
      - then fuzzy match over normalized keys with HIGH cutoff to avoid wrong mappings
        (prevents NETFLIX -> AMAZON)
    """
    if not raw:
        return None, []
    k = norm_service_key(raw)
    if k in NORM_TO_SERVICE:
        return NORM_TO_SERVICE[k], []

    # fuzzy on normalized keys
    candidates = difflib.get_close_matches(k, list(NORM_TO_SERVICE.keys()), n=3, cutoff=fuzzy_cutoff)
    if candidates:
        resolved = NORM_TO_SERVICE[candidates[0]]
        suggestions = [NORM_TO_SERVICE[c] for c in candidates]
        return resolved, suggestions
    return None, []

_ALL_NETWORKS: List[Tuple[object, List[int]]] = []
for pfx, idxs in prefix_to_idxs.items():
    try:
        _ALL_NETWORKS.append((ipaddress.ip_network(pfx, strict=False), idxs))
    except ValueError:
        continue


@dataclass
class Constraints:
    cidr: Optional[str] = None      # exact CIDR from query (e.g., 1.2.3.0/24)
    ip: Optional[str] = None        # IP from query (e.g., 1.2.3.4)
    region: Optional[str] = None
    service: Optional[str] = None   # canonical service as in dataset (e.g., "EC2")
    nbg: Optional[str] = None       # canonical nbg as in dataset (e.g., "internal-restricted")
    service_suggestions: Optional[List[str]] = None


def extract_constraints(question: str) -> Constraints:
    """
    Parses structured identifiers (CIDR/IP/region/service/NBG) directly from query.
    Uses dynamic normalization + safe fuzzy for service token correction.
    """
    q = question.strip()
    c = Constraints(service_suggestions=[])

    # CIDR or IP
    m = CIDR_RE.search(q)
    if m:
        c.cidr = m.group(0).strip()
    else:
        m = IP_RE.search(q)
        if m:
            c.ip = m.group(0).strip()

    # Region
    m = REGION_RE.search(q)
    if m:
        c.region = norm_region_key(m.group(0))

    # NBG
    if NBG_RE.search(q):
        c.nbg = norm_nbg_key("internal-restricted")

    # Service token resolution: scan tokens (supports hyphens too)
    tokens = re.findall(r"[A-Za-z0-9_\-]+", q)
    for t in tokens:
        resolved, sugg = resolve_service_token(t)
        if resolved:
            c.service = resolved
            c.service_suggestions = sugg or []
            break

    return c


def _ip_longest_prefix_match(ip_str: str) -> Optional[str]:
    """
    Return the CIDR (string) that contains ip_str with the greatest prefix length (LPM).
    If none contains it, return None.
    """
    try:
        ip_obj = ipaddress.ip_address(ip_str)
    except ValueError:
        return None

    best = None  # (prefixlen, network)
    for net, _idxs in _ALL_NETWORKS:
        if ip_obj in net:
            cand = (net.prefixlen, net)
            if best is None or cand[0] > best[0]:
                best = cand
    return best[1].with_prefixlen if best else None


def candidate_sets(c: Constraints) -> List[Tuple[str, Set[int]]]:
    """
    Retrieves candidate record sets from exact inverted indexes based on extracted constraints.
    - CIDR: exact match via prefix_to_idxs
    - IP: deterministic containment via ipaddress LPM -> then exact prefix lookup
    - region/service/nbg: exact inverted index lookups
    """
    sets: List[Tuple[str, Set[int]]] = []

    if c.cidr:
        idxs = prefix_to_idxs.get(c.cidr, [])
        if idxs:
            sets.append((f"cidr={c.cidr}", set(idxs)))

    if c.ip:
        lpm = _ip_longest_prefix_match(c.ip)
        if lpm:
            idxs = prefix_to_idxs.get(lpm, [])
            if idxs:
                sets.append((f"ip∈{lpm} (LPM)", set(idxs)))
        else:
            # keep reason for debugging; no set added
            pass

    if c.region:
        idxs = region_to_idxs.get(c.region, [])
        if idxs:
            sets.append((f"region={c.region}", set(idxs)))

    if c.service:
        idxs = service_to_idxs.get(c.service, [])
        if idxs:
            sets.append((f"service={c.service}", set(idxs)))

    if c.nbg:
        idxs = nbg_to_idxs.get(c.nbg, [])
        if idxs:
            sets.append((f"nbg={c.nbg}", set(idxs)))

    return sets


def combine_AND(sets: List[Tuple[str, Set[int]]]) -> Tuple[Optional[Set[int]], List[str]]:
    """Intersects all candidate sets using logical AND to produce a deterministic match set."""
    if not sets:
        return None, []
    out: Optional[Set[int]] = None
    reasons: List[str] = []
    for name, s in sets:
        reasons.append(name)
        out = s if out is None else out & s
    return out, reasons


def _faiss_top1(question: str) -> Tuple[int, float]:
    """Embeds the query and performs cosine similarity search in FAISS to return the most semantically similar record."""
    q_vec = embedder.encode(question, normalize_embeddings=True).astype("float32").reshape(1, -1)
    scores, idxs = index.search(q_vec, k=1)
    return int(idxs[0][0]), float(scores[0][0])

**Layer 3: Hybrid Retrieval Logic (Exact → Controlled Vector Fallback)**
This layer extracts structured constraints from the user query and decides whether to perform deterministic exact retrieval or semantic vector retrieval.

**Constraint extraction**
- Uses regex to detect CIDR or IPv4 patterns, plus region and a known network_border_group token.

- Resolves service names using dynamic normalization (case/spacing/punctuation-insensitive) and an optional high-threshold fuzzy match against the allowlist of known services to tolerate minor typos without mis-mapping unrelated terms.

- Extracted fields are stored in a lightweight Constraints object.

**Deterministic candidate retrieval + AND intersection**
For each extracted constraint, the system retrieves candidate record sets from Layer 1 inverted indexes:

- ip_prefix is matched exactly when a CIDR is present.

- For raw IP inputs, the system performs CIDR containment + Longest Prefix Match (LPM) using Python’s ipaddress module to resolve the IP to the most specific matching prefix, then uses prefix_to_idxs for deterministic lookup.

- service, region, and network_border_group candidates are retrieved from their respective indexes.

- When multiple constraints are present, candidate sets are intersected (AND logic) to produce a deterministic match set.

**Controlled vector fallback**
A FAISS vector search (IndexFlatIP over normalized embeddings) is available as a semantic fallback mode. In this notebook, semantic retrieval is used only when structured constraints are absent, preventing incorrect substitutions for strict identifiers (e.g., returning an unrelated service due to embedding similarity).

**Top-1 behavior**
This implementation returns a single record for a top-1 interface. In production, deterministic matches would return top-k and/or apply ranking (e.g., prefer most-specific prefixes, apply secondary filters, or prompt for clarification when ambiguity remains).

In [189]:
def retrieve_top1(question: str) -> Tuple[Optional[int], float, str]:
    """
    Top-1 retrieval with safe gating.
    Returns: (idx or None, score, mode)

    mode examples:
      - EXACT_AND(region=..., service=...)
      - VECTOR
      - NO_MATCH (unknown service=... | did_you_mean=[...])
      - NO_MATCH (structured constraints but no deterministic match; semantic skipped)
    """
    c = extract_constraints(question)
    sets = candidate_sets(c)
    det_set, reasons = combine_AND(sets)

    had_structured = any([c.cidr, c.ip, c.region, c.service, c.nbg])

    if re.search(r"\bservice\b", question, re.IGNORECASE) and not c.service:

        m = re.search(r"\bservice\s+([A-Za-z0-9_\-]+)\b", question, re.IGNORECASE)
        raw = m.group(1) if m else ""
        _, sugg = resolve_service_token(raw)
        msg = f"NO_MATCH (unknown service={raw})"
        if sugg:
            msg += f" | did_you_mean={sugg}"
        return None, 0.0, msg

    # If structured constraints exist and we formed an AND-set but it's empty -> NO_MATCH (skip FAISS)
    if had_structured and (det_set is not None) and len(det_set) == 0:
        return None, 0.0, "NO_MATCH (structured constraints but no deterministic match; semantic skipped)"

    # If deterministic matches exist, return one (consistent with your original behavior)
    if det_set and len(det_set) > 0:
        idx = min(det_set)
        return idx, 1.0, f"EXACT_AND({', '.join(reasons)})"

    # Only allow semantic fallback when there are NO structured constraints at all
    if not had_structured:
        idx, score = _faiss_top1(question)
        return idx, score, "VECTOR"

    # Structured present but no exact candidate sets were formed (e.g., IP not contained in any prefix)
    return None, 0.0, "NO_MATCH (structured constraints present but no exact candidates; semantic skipped)"

**Core hybrid retrieval function.**

Extracts structured constraints from the query, retrieves candidate records using the deterministic indexes built in Layer 1, and intersects them to produce an exact match set.

If deterministic matches exist, the function returns a top-1 result with full confidence. If structured constraints are present but no deterministic match is found, the system returns NO_MATCH and skips semantic retrieval to avoid incorrect substitutions.

Vector search using FAISS is only used when the query contains no structured constraints, enabling semantic retrieval for purely natural-language queries. The function returns the selected record index, its score, and the retrieval mode used.

In [190]:
def debug_retrieval(question: str):
    idx, score, mode = retrieve_top1(question)

    print("TOP-1 CHUNK:")
    if idx is None:
        print("NONE")
    else:
        print(chunks[idx])

    print("\nSIMILARITY SCORE:")
    print(score)

    # Optional but very useful (and ok to keep)
    print("\nMETHOD:")
    print(mode)

    return idx, score, mode

**Spec-aligned retrieval inspection:**

Calls the hybrid retriever and prints:

- Top-1 chunk

- Similarity score

- Retrieval method

Used to inspect and verify retrieval behavior independently of the QA/LLM pipeline.

In [191]:
def debug_retrieval_verbose(question: str):
    idx, score, mode = retrieve_top1(question)

    print("\n--- VERBOSE TRACE ---")

    cons = extract_constraints(question)
    print("Extracted Constraints:", cons)

    sets = candidate_sets(cons)
    print("Candidate Sets:", sets)

    print("Final Retrieval Method:", mode)

    print("\nTOP-1 CHUNK:")
    if idx is None:
        print("NONE")
    else:
        print(chunks[idx])

    print("\nSCORE:")
    print(score)

**Purpose:**
Detailed tracing utility for the hybrid retriever.

**Prints:**

Extracted constraints (parsed from the query)

Candidate sets (from Layer 1 inverted indexes)

Retrieval mode (EXACT_AND, VECTOR, NO_MATCH)

Top-1 chunk (or NONE)

Score

Used to debug and explain retrieval flow step-by-step.

In [192]:
debug_retrieval("What is the IP address associated with Project Omega?")

TOP-1 CHUNK:
service: PROJECT_OMEGA_UPLINK | ip_prefix: 99.99.99.99/32 | region: us-west-2 | network_border_group: internal-restricted

SIMILARITY SCORE:
0.5598879456520081

METHOD:
VECTOR


(0, 0.5598879456520081, 'VECTOR')

In [193]:
debug_retrieval("99.99.99.99/32")

TOP-1 CHUNK:
service: PROJECT_OMEGA_UPLINK | ip_prefix: 99.99.99.99/32 | region: us-west-2 | network_border_group: internal-restricted

SIMILARITY SCORE:
1.0

METHOD:
EXACT_AND(cidr=99.99.99.99/32)


(0, 1.0, 'EXACT_AND(cidr=99.99.99.99/32)')

Some Testing Samples:

In [194]:
debug_retrieval("What is the IP prefix for AMAZON in us-east-1?")

TOP-1 CHUNK:
service: AMAZON | ip_prefix: 15.230.221.0/24 | region: us-east-1 | network_border_group: us-east-1

SIMILARITY SCORE:
1.0

METHOD:
EXACT_AND(region=us-east-1, service=AMAZON)


(6, 1.0, 'EXACT_AND(region=us-east-1, service=AMAZON)')

In [195]:
debug_retrieval("Show IP prefixes for the service NETFLIX")

TOP-1 CHUNK:
NONE

SIMILARITY SCORE:
0.0

METHOD:
NO_MATCH (unknown service=NETFLIX)


(None, 0.0, 'NO_MATCH (unknown service=NETFLIX)')

In [196]:
debug_retrieval("Show IP prefixes for the 234.67.8.1")

TOP-1 CHUNK:
NONE

SIMILARITY SCORE:
0.0

METHOD:
NO_MATCH (structured constraints present but no exact candidates; semantic skipped)


(None,
 0.0,
 'NO_MATCH (structured constraints present but no exact candidates; semantic skipped)')

In [197]:
debug_retrieval("Tell me about 52.95.245.0/24")

TOP-1 CHUNK:
service: AMAZON | ip_prefix: 52.95.245.0/24 | region: us-east-1 | network_border_group: us-east-1

SIMILARITY SCORE:
1.0

METHOD:
EXACT_AND(cidr=52.95.245.0/24)


(2209, 1.0, 'EXACT_AND(cidr=52.95.245.0/24)')

In [198]:
debug_retrieval("Show me services in us-west-2")

TOP-1 CHUNK:
service: PROJECT_OMEGA_UPLINK | ip_prefix: 99.99.99.99/32 | region: us-west-2 | network_border_group: internal-restricted

SIMILARITY SCORE:
1.0

METHOD:
EXACT_AND(region=us-west-2)


(0, 1.0, 'EXACT_AND(region=us-west-2)')

In [199]:
debug_retrieval_verbose("99.99.99.99")


--- VERBOSE TRACE ---
Extracted Constraints: Constraints(cidr=None, ip='99.99.99.99', region=None, service=None, nbg=None, service_suggestions=[])
Candidate Sets: [('ip∈99.99.99.99/32 (LPM)', {0})]
Final Retrieval Method: EXACT_AND(ip∈99.99.99.99/32 (LPM))

TOP-1 CHUNK:
service: PROJECT_OMEGA_UPLINK | ip_prefix: 99.99.99.99/32 | region: us-west-2 | network_border_group: internal-restricted

SCORE:
1.0


In [200]:
debug_retrieval_verbose("What is the IP address associated with Project Omega?")


--- VERBOSE TRACE ---
Extracted Constraints: Constraints(cidr=None, ip=None, region=None, service=None, nbg=None, service_suggestions=[])
Candidate Sets: []
Final Retrieval Method: VECTOR

TOP-1 CHUNK:
service: PROJECT_OMEGA_UPLINK | ip_prefix: 99.99.99.99/32 | region: us-west-2 | network_border_group: internal-restricted

SCORE:
0.5598879456520081


# Part 3: The QA Chain

In [201]:
!pip install transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LOCAL_MODEL = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_MODEL)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Loaded local LLM:", LOCAL_MODEL, "| device:", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded local LLM: google/flan-t5-base | device: cpu


**Model Choice Rationale**

For this implementation, we selected FLAN-T5 (encoder–decoder) as the local language model.

The decision was guided by the following constraints:

- CPU-friendly – Runs efficiently on CPU without requiring GPU.
- Fully local execution – No external API calls.
- Reproducible – No secret keys, rate limits, or vendor dependencies.
- Deterministic – Sampling disabled for stable evaluation outputs.
- Lightweight setup – Easily executable in Colab or local environments.

While decoder-only models such as LLaMA, Mistral, or other large instruction-tuned models could improve generative quality, they typically require:

- API access or hosted inference
- Secret keys
- GPU resources
- Larger memory footprint

For an assignment setting emphasizing reproducibility and portability, FLAN-T5 provides a reliable, fully self-contained solution.

In [209]:
import time

def call_llm_local(question: str, context: str = "") -> str:
    start = time.perf_counter()

    prompt = (
        "You are a Network Ops assistant.\n"
        "Answer ONLY using the provided context.\n"
        "If the answer isn't in the context, say: I don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    latency = time.perf_counter() - start
    print(f"[LLM Latency for RAG Retrieval] {latency:.4f}s")

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

Generates a grounded answer using a local language model.

The function constructs a prompt instructing the model to answer only using the retrieved context and to respond with “I don't know” when the answer is not present. Generation is performed deterministically (do_sample=False) with a token limit to keep responses concise and consistent for evaluation. This step forms the generation stage of the RAG pipeline, using the retrieved chunk as the model’s context.

In [218]:
def ask_network_bot(question: str) -> str:
    idx, score, mode = retrieve_top1(question)

    # handle NO_MATCH
    if idx is None:
        return "I don't know"

    context_text = chunks[idx]
    ans = call_llm_local(question, context=context_text).strip()

    if (ans or "").strip().lower().replace(".", "").startswith("i don't know"):
        return "I don't know"

    # IMPORTANT: use group(0) and a simple pattern
    m = re.search(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", ans)
    return m.group(0) if m else ans

**Purpose:**
End-to-end QA pipeline.

This function:
- Retrieves the top-1 relevant record using the hybrid retriever.
- Passes the retrieved chunk as context to the LLM.
- Returns the extracted IP address (or "I don't know" if not found).

A regex post-processing step ensures only the IP address is returned, enforcing clean, evaluation-friendly output formatting.

In [220]:
print(ask_network_bot("What is the IP address associated with Project Omega?"))

[LLM Latency for RAG Retrieval] 0.9065s
99.99.99.99


In [221]:
def ask_network_bot_debug(question: str):  # without post processing (demonstrating LLM behavior)
    idx, score, mode = retrieve_top1(question)

    if idx is None:
        print("RAW LLM OUTPUT:")
        print("I don't know")
        print("\nNORMALIZED OUTPUT:")
        print("I don't know")
        return

    context_text = chunks[idx]
    raw_ans = call_llm_local(question, context=context_text).strip()

    print("RAW LLM OUTPUT:")
    print(raw_ans)

    if raw_ans.lower() == "i don't know":
        print("\nNORMALIZED OUTPUT:")
        print("I don't know")
        return

    m = re.search(r"\b(\d{1,3}(?:\.\d{1,3}){3})\b", raw_ans)
    final = m.group(1) if m else raw_ans

    print("\nNORMALIZED OUTPUT:")
    print(final)

In [222]:
ask_network_bot_debug("What is the IP address associated with Project Omega?")

[LLM Latency for RAG Retrieval] 0.8926s
RAW LLM OUTPUT:
99.99.99.99/32

NORMALIZED OUTPUT:
99.99.99.99


**LLM Output Normalization Demonstration**

Large language models may return additional formatting (e.g., CIDR notation /32 or more verbose) even when instructed to output only an IP address.

To ensure deterministic evaluation and contract-compliant output, a regex normalization layer extracts only the IPv4 address.

This block demonstrates the difference between:

Raw LLM output

Post-processed normalized output

In [223]:
import time

def call_llm_unconstrained(question: str) -> str:  # zero shot
    start = time.perf_counter()

    prompt = f"Answer the following question:\n\n{question}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    latency = time.perf_counter() - start
    print(f"[LLM Latency Zero Shot] {latency:.4f}s")

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [225]:
QUESTION = "What is the IP address associated with Project Omega?"

# System A (Zero-shot)
ansA = call_llm_unconstrained(QUESTION)

# System B (RAG)
ansB = ask_network_bot(QUESTION)


def judge(answer: str) -> str:
    answer = (answer or "")
    return "PASS" if "99.99.99.99" in answer else "FAIL"


print("=== System A (Zero-shot) ===")
print(ansA)
print("JUDGE:", judge(ansA))

print("\n=== System B (RAG) ===")
print(ansB)
print("JUDGE:", judge(ansB))

[LLM Latency Zero Shot] 0.5653s
[LLM Latency for RAG Retrieval] 0.9174s
=== System A (Zero-shot) ===
192.168.0.1
JUDGE: FAIL

=== System B (RAG) ===
99.99.99.99
JUDGE: PASS


**System Comparison: Zero-shot vs RAG**

This section compares two approaches for answering the same query.

System A (Zero-shot)
Calls the language model directly without retrieval or external context.

System B (RAG)
Uses the hybrid retrieval pipeline (deterministic structured retrieval with optional semantic fallback) to retrieve the most relevant record, then provides that chunk as context to the LLM.

The judge() function checks whether the expected IP address (99.99.99.99) appears in the model's answer and assigns a PASS/FAIL label.

This evaluation illustrates the effect of retrieval grounding:

Zero-shot relies only on the model’s internal knowledge.

The RAG system provides explicit factual context retrieved from the dataset.

# **Production & Future Improvements**

**Multi-result deterministic resolution**
Currently, deterministic filters return a single top-1 record. In production systems, multiple matching prefixes or services may exist. The system should return top-k candidates, prefer longest-prefix matches, or prompt the user for clarification when ambiguity remains.

**Hybrid ranking improvements**
For semantic fallback queries, add a lightweight cross-encoder reranker after FAISS retrieval to improve precision when multiple semantically similar chunks exist.

**Confidence scoring**
Introduce a calibrated confidence score that combines:

deterministic constraint matches

vector similarity score (when used)

prefix specificity (e.g., longer CIDR prefixes preferred)

This would provide more reliable confidence estimates for downstream systems.

**Evaluation framework**
Replace simple string matching with structured evaluation metrics (e.g., answer correctness, retrieval accuracy, constraint extraction accuracy) and automated benchmarks similar to RAG evaluation frameworks.

**Caching and query optimization**
Cache frequent deterministic lookups (e.g., service/region/IP prefix queries) and repeated vector searches to reduce latency and improve throughput.

**Observability and monitoring**
Add structured logging for:

retrieval mode (EXACT_AND, VECTOR, NO_MATCH)

constraint extraction statistics

retrieval latency

This would support production monitoring and debugging.

**Vector infrastructure upgrades**
For production deployments, replace the in-memory FAISS index with a persistent vector database (e.g., Pinecone or pgvector) to support scaling, persistence, and metadata filtering.